# Power Dominating Set — Path-Cover Heuristic

Pipeline:
1. **Greedy path cover** — repeatedly extract a long path from the remaining graph.
2. **Seed selection per path** — try endpoints + highest-degree interior vertex, simulate each, pick the one that covers the most.
3. **Run real PDS propagation** from all chosen seeds.
4. **Repair** — greedily add seeds until every vertex is covered.

In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import time
import pandas as pd
from collections import deque
from scipy.io import mmread

## 1. PDS propagation simulator

Given a graph `G` and a seed set `S`, return the set of vertices that get dominated by:
- **Rule 1:** every vertex in `S` and all its neighbors are dominated.
- **Rule 2:** if a dominated vertex has exactly one undominated neighbor, that neighbor becomes dominated. Repeat until stable.

In [2]:
def pds_propagate(G, seeds):
    """Return the set of vertices dominated by `seeds` under PDS rules."""
    dominated = set()
    # Rule 1: seeds and their neighbors
    for s in seeds:
        dominated.add(s)
        dominated.update(G.neighbors(s))
    
    # Rule 2: iterative propagation
    changed = True
    while changed:
        changed = False
        # Check every dominated vertex to see if it can propagate
        for v in list(dominated):
            undominated_neighbors = [u for u in G.neighbors(v) if u not in dominated]
            if len(undominated_neighbors) == 1:
                dominated.add(undominated_neighbors[0])
                changed = True
    return dominated

def is_pds(G, S):
    """Check whether S is a valid power dominating set of G."""
    return pds_propagate(G, S) == set(G.nodes())

## 2. Greedy path cover

Repeatedly pick a starting vertex (low degree first — endpoints of natural chains), then extend greedily along unvisited neighbors, preferring low-degree neighbors so we walk through chains instead of jumping into hubs.

In [3]:
def greedy_path_cover(G):
    """Return a list of paths (each a list of vertices) covering all vertices of G."""
    unvisited = set(G.nodes())
    paths = []
    
    while unvisited:
        # Start from the lowest-degree unvisited vertex (often a true endpoint)
        start = min(unvisited, key=lambda v: G.degree(v))
        path = [start]
        unvisited.remove(start)
        
        # Extend forward greedily
        current = start
        while True:
            # Candidate next vertices: unvisited neighbors
            candidates = [u for u in G.neighbors(current) if u in unvisited]
            if not candidates:
                break
            # Prefer low-degree candidates -> stay on chain-like structures
            nxt = min(candidates, key=lambda v: G.degree(v))
            path.append(nxt)
            unvisited.remove(nxt)
            current = nxt
        
        # Try extending backward from the original start too
        current = start
        while True:
            candidates = [u for u in G.neighbors(current) if u in unvisited]
            if not candidates:
                break
            nxt = min(candidates, key=lambda v: G.degree(v))
            path.insert(0, nxt)
            unvisited.remove(nxt)
            current = nxt
        
        paths.append(path)
    return paths

## 3. Seed selection per path (with simulation)

For each path, candidates = the two endpoints + the highest-degree interior vertex. Simulate seeding each one *alone* on the full graph `G` and pick the one that dominates the most vertices.

In [4]:
def pick_seed_for_path(G, path):
    """Choose the best seed vertex from this path by simulation."""
    if len(path) == 1:
        return path[0]
    
    # Candidate set: endpoints + highest-degree interior vertex
    candidates = {path[0], path[-1]}
    if len(path) > 2:
        interior = path[1:-1]
        hi_deg_interior = max(interior, key=lambda v: G.degree(v))
        candidates.add(hi_deg_interior)
    
    # Score each candidate by how many vertices it dominates alone
    best_v, best_score = None, -1
    for v in candidates:
        score = len(pds_propagate(G, {v}))
        # Tiebreak toward higher degree, then toward endpoints
        is_endpoint = (v == path[0] or v == path[-1])
        key = (score, G.degree(v), is_endpoint)
        if best_v is None or key > (best_score, G.degree(best_v),
                                     best_v == path[0] or best_v == path[-1]):
            best_v, best_score = v, score
    return best_v

## 4. Repair phase (greedy)

If propagation from the chosen seeds doesn't cover everything, greedily add the vertex whose inclusion covers the most currently-uncovered vertices. Only consider vertices in or adjacent to the uncovered set — far-away vertices won't help.

In [5]:
def repair(G, seeds):
    """Add seeds greedily until the whole graph is dominated."""
    seeds = set(seeds)
    dominated = pds_propagate(G, seeds)
    
    while dominated != set(G.nodes()):
        uncovered = set(G.nodes()) - dominated
        
        # Candidate pool: uncovered vertices and their neighbors
        candidates = set(uncovered)
        for u in uncovered:
            candidates.update(G.neighbors(u))
        candidates -= seeds  # no point re-adding a seed
        
        best_v, best_gain = None, -1
        for v in candidates:
            new_dominated = pds_propagate(G, seeds | {v})
            gain = len(new_dominated) - len(dominated)
            if gain > best_gain:
                best_gain, best_v = gain, v
        
        if best_v is None or best_gain <= 0:
            # Defensive fallback: add any uncovered vertex
            best_v = next(iter(uncovered))
        
        seeds.add(best_v)
        dominated = pds_propagate(G, seeds)
    
    return seeds

## 5. Putting it all together

In [6]:
def pds_heuristic(G, verbose=False):
    """Path-cover-based heuristic for Power Dominating Set."""
    # Step 1: path cover
    paths = greedy_path_cover(G)
    if verbose:
        print(f"Path cover: {len(paths)} paths")
        for p in paths:
            print(f"  {p}")
    
    # Step 2: pick a seed from each path
    seeds = set()
    for p in paths:
        seeds.add(pick_seed_for_path(G, p))
    if verbose:
        print(f"\nInitial seeds after path-based selection: {seeds}")
        print(f"Coverage after step 3: {len(pds_propagate(G, seeds))}/{G.number_of_nodes()}")
    
    # Step 4: repair
    seeds = repair(G, seeds)
    if verbose:
        print(f"\nFinal PDS: {seeds}  (size {len(seeds)})")
        print(f"Valid PDS? {is_pds(G, seeds)}")
    return seeds

## 6. Test on a few graphs

In [7]:
def load_graph(filepath):
    """
    Load a graph from a Matrix Market (.mtx or .mtx.gz) file.
    Removes self-loops since they are irrelevant to PDS.
    """
    A = mmread(filepath)
    G = nx.from_scipy_sparse_array(A)
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


# Load BCSPWR01 as a test
G = load_graph("datasets/bcspwr01.mtx.gz")

print(f"Nodes         : {G.number_of_nodes()}")
print(f"Edges         : {G.number_of_edges()}")
print(f"Avg degree    : {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")
print(f"Max degree    : {max(G.degree(), key=lambda x: x[1])}")

Nodes         : 39
Edges         : 46
Avg degree    : 2.36
Max degree    : (15, 5)


In [8]:
datasets = [
    ("BCSPWR01", "datasets/bcspwr01.mtx.gz"),
    ("BCSPWR02", "datasets/bcspwr02.mtx.gz"),
    ("BCSPWR03", "datasets/bcspwr03.mtx.gz"),
    ("BCSPWR04", "datasets/bcspwr04.mtx.gz"),
    ("BCSPWR05", "datasets/bcspwr05.mtx.gz"),
    ("BCSPWR06", "datasets/bcspwr06.mtx.gz"),
    ("BCSPWR07", "datasets/bcspwr07.mtx.gz"),
    ("BCSPWR08", "datasets/bcspwr08.mtx.gz"),
    ("BCSPWR09", "datasets/bcspwr09.mtx.gz"),
    ("BCSPWR10", "datasets/bcspwr10.mtx.gz"),
]

results = []

In [9]:
for name, path in datasets:
    try:
        G_i = load_graph(path)

        # Path Cover Heuristic
        t0 = time.time()
        pds_path = pds_heuristic(G_i)
        t_path = time.time() - t0

        results.append({
            "Dataset"          : name,
            "Nodes"            : G_i.number_of_nodes(),
            "Edges"            : G_i.number_of_edges(),
            "PathCover PDS"    : len(pds_path),
            "PathCover Time(s)": round(t_path, 4),
        })

        print(f"{name}: PathCover={len(pds_path)} ({t_path:.4f}s)")

    except FileNotFoundError:
        print(f"{name}: file not found — skipping")

BCSPWR01: PathCover=8 (0.0014s)
BCSPWR02: PathCover=12 (0.0008s)
BCSPWR03: PathCover=21 (0.0349s)
BCSPWR04: PathCover=47 (0.0469s)
BCSPWR05: PathCover=96 (0.3104s)
BCSPWR06: PathCover=321 (1.8075s)
BCSPWR07: PathCover=386 (2.9838s)
BCSPWR08: PathCover=376 (1.7288s)
BCSPWR09: PathCover=388 (3.8003s)


KeyboardInterrupt: 